# Classificating Pizza Brand by Nutrition

# Problem 1


In [49]:
import pandas
import numpy
# import sklearn
data = pandas.read_csv('/content/Pizza (1).csv')
print("Number of samples:", data.shape[0])
print(data.head())

# Cut out feature 'id'
data = data.drop('id', axis=1)
standardized_data = data.copy()

# print(data.head())

# Do a Z-score standardization on features 'mois', 'prot', 'fat', 'ash', 'sodium', 'carb', 'cal'
# features = ['mois', 'prot', 'fat', 'ash', 'sodium', 'carb', 'cal']
# standardized_data.loc[:, features] = (standardized_data.loc[:, features] - standardized_data.loc[:, features].mean()) / standardized_data.loc[:, features].std()
# print(data.head())

# This problem seems to work decently with or without normalization -> I won't be normalizing for the sake of part (c) of this problem

Number of samples: 300
  brand     id   mois   prot    fat   ash  sodium  carb   cal
0     A  14069  27.82  21.43  44.87  5.11    1.77  0.77  4.93
1     A  14053  28.49  21.26  43.89  5.34    1.79  1.02  4.84
2     A  14025  28.35  19.99  45.78  5.08    1.63  0.80  4.95
3     A  14016  30.55  20.15  43.13  4.79    1.61  1.38  4.74
4     A  14005  30.49  21.28  41.65  4.82    1.64  1.76  4.67


## Solving for problem (a)
Find a margin perceptron (hyperplane) that separates brand 'A' from 'B'. Then 'A' from 'C'. Then 'B' from 'C'.

In [50]:
def margin_perceptron(Xk, labels, positive_letter, MAX_ITER=1000, learning_rate=0.1):
  w = numpy.zeros(Xk.shape[1])
  b = 0

  for _ in range(MAX_ITER):
    margin_error = 0

    # Create a randomized order of the dataset each iteration to improve learning rate and reduce potential learning errors
    order = numpy.random.permutation(Xk.shape[0])
    X_tmp = Xk[order]
    labels_tmp = labels[order]

    for i, x in enumerate(X_tmp):
      # Label is still A, B, or C -> need to change this to a 1 or -1
      y = 1 if labels_tmp[i] == positive_letter else -1

      # This is the formula for margin perceptron
      margin_check = 1 - (y * (numpy.dot(w, x) + b))

      # max() function in the slides that lower bounds it to 0
      if margin_check > 0:

        # The worse the error the more weight to put on it
        margin_error += margin_check ** 2
        w += learning_rate * y * x
        b += learning_rate * y

    # If the margin perceptron is found, return value (check for very small value instead of 0 because of funky interactions with floating point numbers)
    if margin_error < 1e-6:
      print("Converged before max iterations.")
      break

  return w, b

def test_margin_perceptron(w, b, test_set, test_labels, positive_letter):
  correct = 0
  total_count = test_set.shape[0]

  for i, x in enumerate(test_set):
    decision = numpy.dot(w, x) + b

    if decision > 0 and test_labels[i] == positive_letter:
      correct += 1
    elif decision < 0 and test_labels[i] != positive_letter:
      correct += 1

  print("Correct Percentage:", (correct / total_count)*100, "%")
  print("Correct Count:", correct)
  print("Total Count:", total_count)
  print("\n")


# Separate the data by 'brand' labels 'A', 'B', and 'C'
A = standardized_data[standardized_data['brand'] == 'A']
B = standardized_data[standardized_data['brand'] == 'B']
C = standardized_data[standardized_data['brand'] == 'C']
# print("A: " + str(A.shape[0]) + " rows")
# print("B: " + str(B.shape[0]) + " rows")
# print("C: " + str(C.shape[0]) + " rows")

# Then separate into training and testing sets
A_train = A.sample(frac=0.75, random_state=0)
A_test = A.drop(A_train.index)
B_train = B.sample(frac=0.75, random_state=0)
B_test = B.drop(B_train.index)
C_train = C.sample(frac=0.75, random_state=0)
C_test = C.drop(C_train.index)
# print("A_train: " + str(A_train.shape[0]) + " rows")
# print("A_test: " + str(A_test.shape[0]) + " rows")

# Separate 'brand' as the label (y) and other features as the variables (x^k)
A_train_y = A_train['brand']
A_train_x = A_train.drop('brand', axis=1)
B_train_y = B_train['brand']
B_train_x = B_train.drop('brand', axis=1)
C_train_y = C_train['brand']
C_train_x = C_train.drop('brand', axis=1)
# print(A_train_x.head())
# print(A_train_y.head())

# Separate 'brand' as the label (y) and other features as the variables (x^k) for the test sets
A_test_y = A_test['brand']
A_test_x = A_test.drop('brand', axis=1)
B_test_y = B_test['brand']
B_test_x = B_test.drop('brand', axis=1)
C_test_y = C_test['brand']
C_test_x = C_test.drop('brand', axis=1)

# Feed 'A' and 'B' in first, concatenate A_train_x to B_train_x and A_train_y to B_train_y -> this gets randomly shuffled at the beginning of each iteration so the order here doesn't matter
tmp_x = numpy.concatenate((A_train_x.to_numpy(), B_train_x.to_numpy()))
tmp_y = numpy.concatenate((A_train_y.to_numpy(), B_train_y.to_numpy()))
w_AB, b_AB = margin_perceptron(tmp_x, tmp_y, 'A')
print("Margin Perceptron between A and B: " + str(w_AB) + " x^k + (" + str(b_AB) + ")")
# Test the margin perceptron using the samples set aside for testing 'A' and 'B'
tmp_x = numpy.concatenate((A_test_x.to_numpy(), B_test_x.to_numpy()))
tmp_y = numpy.concatenate((A_test_y.to_numpy(), B_test_y.to_numpy()))
print("Test:")
test_margin_perceptron(w_AB, b_AB, tmp_x, tmp_y, 'A')

# Feed 'A' and 'C' in next
tmp_x = numpy.concatenate((A_train_x.to_numpy(), C_train_x.to_numpy()))
tmp_y = numpy.concatenate((A_train_y.to_numpy(), C_train_y.to_numpy()))
w_AC, b_AC = margin_perceptron(tmp_x, tmp_y, 'A')
print("Margin Perceptron between A and C: " + str(w_AC) + " x^k + (" + str(b_AC) + ")")
# Test the margin perceptron using the samples set aside for testing 'A' and 'C'
tmp_x = numpy.concatenate((A_test_x.to_numpy(), C_test_x.to_numpy()))
tmp_y = numpy.concatenate((A_test_y.to_numpy(), C_test_y.to_numpy()))
print("Test:")
test_margin_perceptron(w_AC, b_AC, tmp_x, tmp_y, 'A')

# Feed 'B' and 'C' in last
tmp_x = numpy.concatenate((B_train_x.to_numpy(), C_train_x.to_numpy()))
tmp_y = numpy.concatenate((B_train_y.to_numpy(), C_train_y.to_numpy()))
w_BC, b_BC = margin_perceptron(tmp_x, tmp_y, 'B')
print("Margin Perceptron between B and C: " + str(w_BC) + " x^k + (" + str(b_BC) + ")")
# Test the margin perceptron using the samples set aside for testing 'B' and 'C'
tmp_x = numpy.concatenate((B_test_x.to_numpy(), C_test_x.to_numpy()))
tmp_y = numpy.concatenate((B_test_y.to_numpy(), C_test_y.to_numpy()))
print("Test:")
test_margin_perceptron(w_BC, b_BC, tmp_x, tmp_y, 'B')

Converged before max iterations.
Margin Perceptron between A and B: [-5.098  1.559  3.663  0.419  0.18  -0.543  0.371] x^k + (0.0)
Test:
Correct Percentage: 100.0 %
Correct Count: 15
Total Count: 15


Converged before max iterations.
Margin Perceptron between A and C: [-2.113 -0.556  2.56   0.13   0.119 -0.021  0.207] x^k + (0.0)
Test:
Correct Percentage: 100.0 %
Correct Count: 14
Total Count: 14


Converged before max iterations.
Margin Perceptron between B and C: [ 0.285 -2.081  1.383  0.091  0.105  0.322  0.054] x^k + (0.0)
Test:
Correct Percentage: 100.0 %
Correct Count: 15
Total Count: 15




## Solving for problem (b)
Compute the margins provided by the linear classifiers you found in part (a).

In [51]:
def find_real_margin(w, b, positive_set, negative_set):
  w_norm = numpy.linalg.norm(w)
  pos_distances = numpy.abs(numpy.dot(positive_set, w) + b) / w_norm
  neg_distances = numpy.abs(numpy.dot(negative_set, w) + b) / w_norm
  positive_min = numpy.min(pos_distances)
  negative_min = numpy.min(neg_distances)
  return positive_min, negative_min

# Use the whole set, go back and drop 'brand'
tmp_A = A.drop('brand', axis=1).to_numpy()
tmp_B = B.drop('brand', axis=1).to_numpy()
tmp_C = C.drop('brand', axis=1).to_numpy()

# Go through each data point in 'A' and 'B', find the smallest margin to 'A' and smallest margin to 'B'
pos_AB, neg_AB = find_real_margin(w_AB, b_AB, tmp_A, tmp_B)
print("For the hyperplane between brand 'A' and brand 'B'")
print("The margin to A is", pos_AB)
print("The margin to B is", neg_AB, "\n")

# Go through each data point in 'A' and 'C', find the smallest margin to 'A' and smallest margin to 'C'
pos_AC, neg_AC = find_real_margin(w_AC, b_AC, tmp_A, tmp_C)
print("For the hyperplane between brand 'A' and brand 'C'")
print("The margin to A is", pos_AC)
print("The margin to C is", neg_AC, "\n")

# Go through each data point in 'B' and 'C', find the smallest margin to 'B' and smallest margin to 'C
pos_BC, neg_BC = find_real_margin(w_BC, b_BC, tmp_B, tmp_C)
print("For the hyperplane between brand 'B' and brand 'C'")
print("The margin to B is", pos_BC)
print("The margin to C is", neg_BC, "\n")

For the hyperplane between brand 'A' and brand 'B'
The margin to A is 0.5735215735244061
The margin to B is 17.657943275363042 

For the hyperplane between brand 'A' and brand 'C'
The margin to A is 6.865747919140207
The margin to C is 14.791746604037524 

For the hyperplane between brand 'B' and brand 'C'
The margin to B is 6.750074735710234
The margin to C is 0.76836393141855 



## Solving for problem (c)
Find the fusion rule for classifying brands 'A', 'B', and 'C'. Label the following instances using the fusion rule.

> s1 =  (49.29, 24.82, 21.68, 2.76, 0.52, 1.47, 3.00)\
> s2 = (30.95, 19.81, 42.28, 5.11, 1.67, 1.85, 4.67)\
> s3 = (50.33, 13.28, 28.43, 3.58, 1.03, 4.38, 3.27)

In [52]:
# These are the 3 rows we have to classify
s1 = [49.29, 24.82, 21.68, 2.76, 0.52, 1.47, 3.00]
s2 = [30.95, 19.81, 42.28, 5.11, 1.67, 1.85, 4.67]
s3 = [50.33, 13.28, 28.43, 3.58, 1.03, 4.38, 3.27]
# Retrieve the datasets that are not normalized
A = data[data['brand'] == 'A']
B = data[data['brand'] == 'B']
C = data[data['brand'] == 'C']

# Now the fusion rule as stated in the slides is argmax_{j=1,...,c} w^T_j x + b_j
# I assume we want to build off of parts (a) and (b) so this would call for a one versus one multiclass classification

def quick_label(num):
  match num:
    case 0:
      return 'A'
    case 1:
      return 'B'
    case 2:
      return 'C'

def fusion_rule(w_AB, b_AB, w_AC, b_AC, w_BC, b_BC, x):
  # Labels: A, B, C
  votes = [0, 0, 0]
  decision = numpy.dot(w_AB, x) + b_AB
  if decision > 0:
    votes[0] += 1
  else:
    votes[1] += 1

  decision = numpy.dot(w_AC, x) + b_AC
  if decision > 0:
    votes[0] += 1
  else:
    votes[2] += 1

  decision = numpy.dot(w_BC, x) + b_BC
  if decision > 0:
    votes[1] += 1
  else:
    votes[2] += 1

  print("Votes:")
  print("A:", votes[0])
  print("B:", votes[1])
  print("C:", votes[2])
  print("Label:", quick_label(numpy.argmax(votes)))
  print("\n")

fusion_rule(w_AB, b_AB, w_AC, b_AC, w_BC, b_BC, s1)
fusion_rule(w_AB, b_AB, w_AC, b_AC, w_BC, b_BC, s2)
fusion_rule(w_AB, b_AB, w_AC, b_AC, w_BC, b_BC, s3)

Votes:
A: 0
B: 1
C: 2
Label: C


Votes:
A: 2
B: 1
C: 0
Label: A


Votes:
A: 0
B: 2
C: 1
Label: B


